# Here In it, we have to perform:

Handle missing values (already done)
Feature engineering
Train-test split
StandardScaler
Sequence generation for LSTM 

In [54]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

In [55]:
df = pd.read_csv(
    "household_power_consumption.txt",
    sep=";",
    na_values="?",
    low_memory=False
)

In [56]:
numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [57]:
df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S"
)

df.set_index("Datetime", inplace=True)

In [58]:
df.drop(columns=["Date", "Time"], inplace=True)

In [59]:
df.ffill(inplace=True)

In [60]:
## Feature Extraction
df["Hour"] = df.index.hour
df["Day"] = df.index.day
df["Month"] = df.index.month
df["Year"] = df.index.year
df["Weekday"] = df.index.dayofweek

In [61]:
target = "Global_active_power"
features = df.columns.tolist()
features.remove(target)

In [62]:
X = df[features]
y = df[target]

In [63]:
# from sklearn.model_selection import train_test_split
# x_train, x_test, y_train, y_test = train_test_split(
#     x, y, test_size=0.2, random_state=42)
# for a time series we dont use train_test_split because it suffles the data
train_size = int(len(df) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

In [64]:
# Standardize the features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [65]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

In [66]:
## Create sliding windows for time series forecasting
# it helps to predct next value
TIME_STEPS = 60

def create_sequences(X, y, time_steps):
    Xs = []
    ys = []

    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:i+time_steps].values)
        ys.append(y.iloc[i+time_steps])

    return np.array(Xs), np.array(ys)

In [67]:
import torch
from torch.utils.data import Dataset

class PowerDataset(Dataset):
    def __init__(self, X, y, seq_length):
        self.X = X.values.astype("float32")
        self.y = y.values.astype("float32")
        self.seq_length = seq_length

    def __len__(self):
        return len(self.X) - self.seq_length

    def __getitem__(self, idx):
        x = self.X[idx:idx+self.seq_length]
        y = self.y[idx+self.seq_length]

        return torch.tensor(x), torch.tensor(y)

In [68]:
SEQ_LENGTH = 60

train_dataset = PowerDataset(
    X_train_scaled,
    y_train,
    SEQ_LENGTH
)

test_dataset = PowerDataset(
    X_test_scaled,
    y_test,
    SEQ_LENGTH
)

In [69]:
from torch.utils.data import DataLoader

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [70]:
print(len(train_dataset))
print(len(test_dataset))

1660147
414992


In [71]:
X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([128, 60, 11])
torch.Size([128])


In [73]:
# saving the scaler for future use
import os
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']